# 09. Final Validation Framework Results

This notebook consolidates the validation framework we will use for the revised main body.

Final framework:

1. **Model configurations remain locked** from `06_final_report_model_lock.ipynb`.
2. **Primary evidence** uses models trained through 2014 and evaluated on a pooled 2018 + 2022 holdout.
3. **Reason for pooling 2018 + 2022:** using only one World Cup is too brittle; 2022 was a winter tournament and may be less representative of normal World Cup timing, while pooling improves sample size and avoids over-interpreting a single tournament.
4. **Uncertainty** is quantified with paired bootstrap intervals on the pooled holdout.
5. **Temporal leakage concern** is handled by replacing LOTO as primary validation evidence with an expanding-window forward-validation sensitivity check.
6. **2018 vs 2022 split** is used as a diagnostic, not as the main test.

This notebook should be the source for main-body revisions.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 160)

NOTEBOOK_DIR = Path.cwd()
LOCK_DIR = NOTEBOOK_DIR / "data" / "final_report_lock"
UNCERTAINTY_DIR = NOTEBOOK_DIR / "data" / "uncertainty_audit"
CV_DIR = NOTEBOOK_DIR / "data" / "cv_leakage_audit"
OUT_DIR = NOTEBOOK_DIR / "data" / "final_validation_framework"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_KEYS = ["baseline", "team_history", "player_enriched"]
MODEL_NAMES = {
    "baseline": "Baseline",
    "team_history": "Team-history",
    "player_enriched": "Player-enriched",
}
CLASS_NAMES = {0: "fav_loses", 1: "draw", 2: "fav_wins"}

print("LOCK_DIR:", LOCK_DIR)
print("UNCERTAINTY_DIR:", UNCERTAINTY_DIR)
print("CV_DIR:", CV_DIR)
print("OUT_DIR:", OUT_DIR)

## 1. Load Existing Locked/Audit Results

These files were generated by:

- `06_final_report_model_lock.ipynb`
- `07_uncertainty_and_cv_audit.ipynb`
- `08_temporal_cv_leakage_sensitivity.ipynb`

In [ ]:
holdout = pd.read_parquet(LOCK_DIR / "holdout_predictions.parquet")
holdout["stage"] = np.where(holdout["knockout_stage"].astype(bool), "knockout_stage", "group_stage")
holdout["actual"] = holdout["y"].map(CLASS_NAMES)

table1 = pd.read_csv(LOCK_DIR / "table1_summary.csv")
overall_uncertainty = pd.read_csv(UNCERTAINTY_DIR / "table1_overall_uncertainty.csv")
overall_differences = pd.read_csv(UNCERTAINTY_DIR / "table1_pairwise_differences.csv")
stage_uncertainty = pd.read_csv(UNCERTAINTY_DIR / "stage_uncertainty.csv")
stage_differences = pd.read_csv(UNCERTAINTY_DIR / "stage_pairwise_differences.csv")
context_baselines = pd.read_csv(UNCERTAINTY_DIR / "context_baselines.csv")
upset_summary = pd.read_csv(UNCERTAINTY_DIR / "upset_aggregate_summary.csv")
conf_uncertainty = pd.read_csv(UNCERTAINTY_DIR / "confederation_accuracy_uncertainty.csv")

forward_summary = pd.read_csv(CV_DIR / "option_a_forward_validation_summary.csv")
forward_folds = pd.read_csv(CV_DIR / "option_a_forward_validation_folds.csv")
loto_leakage = pd.read_csv(CV_DIR / "current_loto_leakage_audit.csv")

print("Holdout rows:", len(holdout))
display(holdout.groupby(["year", "stage"]).size().to_frame("n_matches"))
display(holdout["actual"].value_counts().rename("n").to_frame())

## 2. Primary Pooled Holdout Results with Uncertainty

This is the main evidence table for the revised report. The point estimates are the current Table 1 values; the intervals come from paired bootstrap resampling of the pooled 2018 + 2022 holdout.

In [ ]:
primary_rows = []
for model_key in MODEL_KEYS:
    row = table1[table1["model_key"] == model_key].iloc[0]
    acc = overall_uncertainty[
        (overall_uncertainty["model_key"] == model_key) & (overall_uncertainty["metric"] == "accuracy")
    ].iloc[0]
    f1 = overall_uncertainty[
        (overall_uncertainty["model_key"] == model_key) & (overall_uncertainty["metric"] == "macro_f1")
    ].iloc[0]
    primary_rows.append({
        "model_key": model_key,
        "model_name": MODEL_NAMES[model_key],
        "training_window": row["training_window"],
        "main_signals": row["main_signals"],
        "cv_accuracy_loto_current_not_primary": row["cv_accuracy"],
        "holdout_accuracy": row["holdout_accuracy"],
        "holdout_accuracy_ci_low": acc["ci_low"],
        "holdout_accuracy_ci_high": acc["ci_high"],
        "holdout_macro_f1": row["holdout_macro_f1"],
        "holdout_macro_f1_ci_low": f1["ci_low"],
        "holdout_macro_f1_ci_high": f1["ci_high"],
    })

primary_holdout_table = pd.DataFrame(primary_rows)
primary_holdout_table.to_csv(OUT_DIR / "primary_pooled_holdout_with_uncertainty.csv", index=False)
display(primary_holdout_table.round(3))

## 3. Paired Differences That Drive the Main Claims

Positive values favor the player-enriched model over the team-history model.

In [ ]:
claim_differences = pd.concat([
    overall_differences.assign(scope="overall"),
    stage_differences.assign(scope="stage"),
], ignore_index=True)

claim_differences = claim_differences[[
    "scope", "stage", "comparison", "metric", "n", "point_difference", "ci_low", "ci_high", "minus_2sd", "plus_2sd",
]]
claim_differences.to_csv(OUT_DIR / "player_minus_team_history_claim_differences.csv", index=False)
display(claim_differences.round(3))

## 4. 2018 vs 2022 Diagnostic

These values use the same locked models trained through 2014 and split the pooled holdout by tournament year. We will use this only as a diagnostic because 2022 was a winter World Cup and may be distribution-shifted.

In [ ]:
def metric_row(df, model_key, group_label):
    return {
        **group_label,
        "model_key": model_key,
        "model_name": MODEL_NAMES[model_key],
        "n": int(len(df)),
        "accuracy": accuracy_score(df["y"], df[f"{model_key}_pred"]),
        "macro_f1": f1_score(df["y"], df[f"{model_key}_pred"], average="macro", zero_division=0),
        "n_fav_loses": int((df["y"] == 0).sum()),
        "n_draw": int((df["y"] == 1).sum()),
        "n_fav_wins": int((df["y"] == 2).sum()),
    }


year_rows = []
for year, sub_year in holdout.groupby("year"):
    for model_key in MODEL_KEYS:
        year_rows.append(metric_row(sub_year, model_key, {"year": int(year), "stage": "overall"}))
    for stage, sub_stage in sub_year.groupby("stage"):
        for model_key in MODEL_KEYS:
            year_rows.append(metric_row(sub_stage, model_key, {"year": int(year), "stage": stage}))

year_stage_diagnostic = pd.DataFrame(year_rows)
year_stage_diagnostic.to_csv(OUT_DIR / "holdout_2018_2022_year_stage_diagnostic.csv", index=False)
display(year_stage_diagnostic.round(3))

## 5. Leakage-Safe Expanding-Window Sensitivity

This replaces LOTO as the validation check we discuss in the main body or methodology note. LOTO can be described in the appendix as a stability diagnostic, but it should not be the headline validation evidence because most folds train on future tournaments.

In [ ]:
forward_summary_out = forward_summary.copy()
forward_summary_out.to_csv(OUT_DIR / "expanding_window_sensitivity_summary.csv", index=False)
forward_folds.to_csv(OUT_DIR / "expanding_window_sensitivity_folds.csv", index=False)

leakage_summary = (
    loto_leakage.groupby("model_name")["uses_future_tournaments"]
    .agg(folds_using_future_tournaments="sum", total_loto_folds="count")
    .reset_index()
)
leakage_summary.to_csv(OUT_DIR / "loto_leakage_summary.csv", index=False)

display(forward_summary_out.round(3))
display(leakage_summary)

## 6. Contextualization and Known Weaknesses

These are the numeric anchors for the report's limitations and contextualization paragraphs.

In [ ]:
context_baselines.to_csv(OUT_DIR / "context_baselines_for_report.csv", index=False)
upset_summary.to_csv(OUT_DIR / "upset_summary_for_report.csv", index=False)
conf_uncertainty.to_csv(OUT_DIR / "confederation_uncertainty_for_report.csv", index=False)

display(context_baselines.round(3))
display(upset_summary.round(3))
display(conf_uncertainty[[
    "confederation_code", "model_name", "n", "point_estimate", "ci_low", "ci_high",
]].round(3))

## 7. Result-Based Implications for Main Body Revision

Use these as the guardrails for the rewrite:

1. **Overall model ranking:** do not claim player-enriched is globally better. Its overall accuracy ties the baseline and is only 0.8 percentage points above team-history, with an interval crossing zero.
2. **Macro-F1:** player-enriched has the highest point estimate, but the paired overall macro-F1 difference versus team-history also crosses zero.
3. **Knockout claim:** strongest positive signal for player-enriched, but still uncertainty-limited because the pooled knockout sample is only 32 matches.
4. **Group stage:** no evidence of player-data improvement.
5. **2022 caveat:** 2022 alone does not support a player-enriched advantage, which reinforces using pooled holdout and cautious language.
6. **CV/leakage:** LOTO should not be presented as primary validation. Use expanding-window as leakage-safe sensitivity and pooled 2018 + 2022 as primary holdout.
7. **Context baseline:** majority-favorite accuracy is 66.4%, which means accuracy alone is not a persuasive success metric.